# 01 - Bronze Layer: Salesforce Ingestion

## What this notebook does
Reads the JSON Lines files landed by the `salesforce_incremental_extract` Airflow
DAG out of `raw_data/salesforce/<object>/` and appends them, as-is, into one bronze
Delta table per Salesforce object, using Databricks Autoloader (`cloudFiles`) with
`trigger(availableNow=True)` - the same incremental, checkpoint-tracked pattern as
Project 1's bronze notebook, so re-running this does not reprocess files already
loaded.

## Why bronze keeps everything, including deletes
Bronze is a faithful copy of what Airflow landed: soft-deleted Salesforce records
(`IsDeleted = true`) and any duplicate landings from a watermark boundary are kept
here on purpose. Filtering deletes and de-duplicating by `Id`/`SystemModstamp` is
silver's job, not bronze's.

## Why JSON Lines, not CSV, for this source
Unlike Projects 1-4's Kaggle CSVs, this source has deliberately blank fields
(`LeadSource`, `Amount`) that need to stay real nulls, not empty strings, for
silver to handle correctly. JSON preserves that distinction; CSV does not.

## Tables created
- `main.sales_revenue_analytics.bronze_account`
- `main.sales_revenue_analytics.bronze_contact`
- `main.sales_revenue_analytics.bronze_lead`
- `main.sales_revenue_analytics.bronze_opportunity`
- `main.sales_revenue_analytics.bronze_campaign`

In [0]:
from pyspark.sql import functions as F

CATALOG = "main"
SCHEMA = "sales_revenue_analytics"
VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/raw_data/salesforce"
CHECKPOINT_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/checkpoints"

OBJECTS = ["Account", "Contact", "Lead", "Opportunity", "Campaign"]

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")

DataFrame[]

In [0]:
def ingest_object(sf_object: str) -> dict:
    """Autoload every new JSON Lines file for one Salesforce object into its bronze table.
    Uses trigger(availableNow=True) so it behaves like a batch job: it processes
    whatever is new since the last run, then stops - the same reasoning as Project 1's
    Autoloader function, just applied to JSON files from Airflow instead of CSVs."""
    obj_lower = sf_object.lower()
    raw_path = f"{VOLUME_ROOT}/{obj_lower}/"
    checkpoint_path = f"{CHECKPOINT_ROOT}/bronze_{obj_lower}/checkpoint"
    schema_path = f"{CHECKPOINT_ROOT}/bronze_{obj_lower}/schema"
    table_name = f"{CATALOG}.{SCHEMA}.bronze_{obj_lower}"

    before = (
        spark.table(table_name).count()
        if spark.catalog.tableExists(table_name)
        else 0
    )

    df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "json")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .load(raw_path)
        .withColumn("_bronze_object", F.lit(sf_object))
        .withColumn("_bronze_ingested_at", F.current_timestamp())
        .withColumn("_bronze_source_file", F.col("_metadata.file_path"))
    )

    query = (
        df.writeStream.format("delta")
        .option("checkpointLocation", checkpoint_path)
        .outputMode("append")
        .trigger(availableNow=True)
        .toTable(table_name)
    )
    query.awaitTermination()

    after = spark.table(table_name).count()
    return {"object": sf_object, "table": table_name,
            "rows_before": before, "rows_after": after,
            "rows_added_this_run": after - before}

In [0]:
results = [ingest_object(obj) for obj in OBJECTS]

print(f"{'object':<12}{'rows_before':>14}{'rows_after':>14}{'added_this_run':>18}")
for r in results:
    print(f"{r['object']:<12}{r['rows_before']:>14}{r['rows_after']:>14}{r['rows_added_this_run']:>18}")

object         rows_before    rows_after    added_this_run
Account                  0            78                78
Contact                  0            20                20
Lead                     0           535               535
Opportunity              0           447               447
Campaign                 0             4                 4


In [0]:
%sql
SELECT 'bronze_account' AS table_name, COUNT(*) AS row_count FROM main.sales_revenue_analytics.bronze_account
UNION ALL
SELECT 'bronze_contact', COUNT(*) FROM main.sales_revenue_analytics.bronze_contact
UNION ALL
SELECT 'bronze_lead', COUNT(*) FROM main.sales_revenue_analytics.bronze_lead
UNION ALL
SELECT 'bronze_opportunity', COUNT(*) FROM main.sales_revenue_analytics.bronze_opportunity
UNION ALL
SELECT 'bronze_campaign', COUNT(*) FROM main.sales_revenue_analytics.bronze_campaign;

table_name,row_count
bronze_account,78
bronze_contact,20
bronze_lead,535
bronze_opportunity,447
bronze_campaign,4


In [0]:
%sql
SELECT COUNT(*) AS deleted_rows FROM main.sales_revenue_analytics.bronze_opportunity WHERE IsDeleted = true;

deleted_rows
34


In [0]:
%sql

SELECT Id, LeadSource, Amount FROM main.sales_revenue_analytics.bronze_opportunity
WHERE LeadSource IS NULL OR Amount IS NULL
LIMIT 10;

Id,LeadSource,Amount
006ak00000bTN7BAAW,null,1000.0
006ak00000aNeWkAAK,null,675000.0
006ak00000aNeWlAAK,null,35000.0
006ak00000aNeWIAA0,null,125000.0
006ak00000aNeWLAA0,null,15000.0
006ak00000aNeWMAA0,null,270000.0
006ak00000aNeWTAA0,null,60000.0
006ak00000aNeWVAA0,null,270000.0
006ak00000bTDy9AAG,null,83000.0
006ak00000bTDyOAAW,Partner,null


In [0]:
%sql
SELECT IsDeleted, Description, COUNT(*) AS n
FROM main.sales_revenue_analytics.bronze_opportunity
WHERE LeadSource IS NULL
GROUP BY IsDeleted, Description
ORDER BY IsDeleted, Description;

IsDeleted,Description,n
false,Synthetic demo data (seed 42),15
true,null,8


In [0]:
%sql
SELECT Id, LeadSource, Amount, Description
FROM main.sales_revenue_analytics.bronze_opportunity
WHERE IsDeleted = false AND Description LIKE '%Synthetic demo data%'
  AND (LeadSource IS NULL OR Amount IS NULL)
ORDER BY Id
LIMIT 10;

Id,LeadSource,Amount,Description
006ak00000bTDy9AAG,null,83000.0,Synthetic demo data (seed 42)
006ak00000bTDyOAAW,Partner,null,Synthetic demo data (seed 42)
006ak00000bThAnAAK,null,68500.0,Synthetic demo data (seed 42)
006ak00000bThBLAA0,null,41000.0,Synthetic demo data (seed 42)
006ak00000bThBOAA0,Employee Referral,null,Synthetic demo data (seed 42)
006ak00000bThBaAAK,null,50500.0,Synthetic demo data (seed 42)
006ak00000bThBiAAK,null,466000.0,Synthetic demo data (seed 42)
006ak00000bThBsAAK,null,196500.0,Synthetic demo data (seed 42)
006ak00000bThC1AAK,null,65500.0,Synthetic demo data (seed 42)
006ak00000bThEVAA0,null,211000.0,Synthetic demo data (seed 42)
